# **Mount Drive**

# **Load Data**

In [ ]:
import pandas as pd
import numpy as np

fpath_yield = '<DATA_ROOT>/Yield/crops_us_yield_agg_irig.csv'
dfy = pd.read_csv(fpath_yield)
df_grouped = dfy.groupby(['state_name','county_name','prodn_practice_desc','commodity_desc'])

def detrend_yield(yield_series):
  detrended_yield = np.zeros(len(yield_series))
  if len(yield_series) > 3:
    arr = np.array(yield_series)
    rolling_avg = pd.Series(arr).rolling(window=5,min_periods=1,center=False).mean()
    detrended_yield = np.array(rolling_avg)
  return pd.Series(detrended_yield, index=yield_series.index)

dfy['detrended_yield'] = df_grouped['value_yield'].transform(detrend_yield)
dfy = dfy[dfy['detrended_yield'] > 0]
dfy['d_yield_standard'] = (dfy['value_yield'] - dfy['detrended_yield']) / dfy['detrended_yield']

fpath_fips = '<DATA_ROOT>/Yield/yield_fips_code.csv'
dffipsy = pd.read_csv(fpath_fips).rename(columns={' county_code':'county_code'})
df_new = dffipsy.copy()
formatted_numbers = [str(n).zfill(3) for n in df_new['county_code'].values]
df_new['county_code'] = formatted_numbers
formatted_numbers = [str(n).zfill(2) for n in df_new['state_fips_code'].values]
df_new['state_fips_code'] = formatted_numbers
df_new['FIPS'] = df_new['state_fips_code'] + df_new['county_code']
dffipsy = df_new
dffipsy = dffipsy.drop(columns=['state_fips_code' , 'county_code'])
dffipsy

dfy = pd.merge(dfy,dffipsy,on=['state_name', 'county_name'])

cols = ['FIPS','commodity_desc','prodn_practice_desc','year','d_yield_standard']
df_yield = dfy[cols]
condy = df_yield.year > 2008
df_yield = df_yield[condy]
# df_yield
df_yield.groupby('prodn_practice_desc').count()

<ipython-input-41-2b70f924c4b5>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfy['d_yield_standard'] = (dfy['value_yield'] - dfy['detrended_yield']) / dfy['detrended_yield']


,FIPS,commodity_desc,year,d_yield_standard
prodn_practice_desc,,,,
ALL PRODUCTION PRACTICES,86666,86666,86666,86666
IRRIGATED,2924,2924,2924,2924
NON-IRRIGATED,3394,3394,3394,3394


In [ ]:
# cond1 = dfy.prodn_practice_desc == "ALL PRODUCTION PRACTICES"
# cond2 = dfy.commodity_desc == "WHEAT"
# cond3 = dfy.state_name == "WYOMING"
# cond4 = dfy.county_name == "WESTON"

# temp = dfy[cond1 & cond2 & cond3 & cond4][['year','value_yield','detrended_yield','d_yield_standard']]
# temp = temp.set_index('year')
# temp.plot(y=['value_yield','detrended_yield','d_yield_standard'], figsize=(10, 6))


In [ ]:
from glob import glob
import pandas as pd
in_path = '<DATA_ROOT>/SVI_weightedCrop_modified_Final/*.csv'
files = glob(in_path)

list_df = []
for f in files:
  crop = f.split('/')[-1].split('.')[0][4:]
  # print(crop)
  dfi = pd.read_csv(f).copy()
  dfi['EPL_PCI'] = dfi['EPL_PCI'].fillna(dfi['EPL_HBURD'])
  dfi = dfi.drop(columns=['EPL_HBURD' ,	'EPL_UNINSUR'])
  dfi['Crop'] = crop
  list_df.append(dfi)

df_svi = pd.concat(list_df,ignore_index=True)
df_svi = df_svi.set_index(['STCNTY','YEAR','Crop']).dropna(how='all').reset_index()
df_svi = df_svi.rename(columns={'STCNTY':'FIPS','YEAR':'year'})
df_svi

,FIPS,year,Crop,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,RPL_THEME1,EPL_AGE65,EPL_AGE17,...,EPL_MINRTY,EPL_LIMENG,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,RPL_THEME4,RPL_THEMES
0,1009,2008,HAY,0.411000,0.269000,0.555000,0.623000,0.460000,0.394000,0.551000,...,0.154000,0.039000,0.002000,0.000000,0.947000,0.087000,0.174000,0.714000,0.458000,0.342000
1,1015,2008,HAY,0.440000,0.502000,0.430000,0.409000,0.431000,0.567000,0.293000,...,0.512000,0.149000,0.287000,0.435000,0.761000,0.050000,0.291000,0.000000,0.285000,0.322000
2,1043,2008,HAY,0.602000,0.376400,0.674400,0.718400,0.638000,0.541800,0.443200,...,0.069000,0.195600,0.069600,0.045800,0.916200,0.203400,0.471800,0.000000,0.270200,0.369200
3,1083,2008,HAY,0.562800,0.329280,0.555440,0.834680,0.610120,0.283880,0.512200,...,0.628960,0.331480,0.486560,0.012440,0.872000,0.522080,0.520000,0.039680,0.400400,0.496320
4,1089,2008,HAY,0.414111,0.390333,0.205222,0.235889,0.241000,0.192444,0.441667,...,0.557556,0.372444,0.469778,0.553111,0.706889,0.209778,0.168556,0.081111,0.310667,0.216556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279115,56037,2022,OATS,0.421756,0.641478,0.103969,0.282411,0.333217,0.673786,0.417695,...,0.147250,0.454942,0.147250,0.029158,0.956544,0.487839,0.138129,0.426352,0.420015,0.434108
279116,56039,2022,OATS,0.247541,0.171324,0.273629,0.168196,0.163031,0.507233,0.262789,...,0.276512,0.444131,0.276512,0.536832,0.651444,0.564276,0.410931,0.619973,0.709801,0.280541
279117,56041,2022,OATS,0.532133,0.641156,0.090844,0.394458,0.480949,0.363293,0.836505,...,0.268444,0.446553,0.268444,0.552860,0.968887,0.800458,0.332027,0.500506,0.777016,0.629125
279118,56043,2022,OATS,0.440896,0.350872,0.080058,0.266097,0.298983,0.784543,0.526574,...,0.135750,0.009997,0.135750,0.340103,0.828429,0.819263,0.402957,0.796382,0.855720,0.455858


In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_v3.csv'
dfw = pd.read_csv(fpath_water).rename(columns={'Year':'year'})
dfw = dfw[['FIPS','IR-WGWFr','IR-WSWFr','IR-WFrTo','IR-IrTot','year']]
dfw['IR-WGWFr'] = pd.to_numeric(dfw['IR-WGWFr'], errors='coerce')
dfw['IR-WSWFr'] = pd.to_numeric(dfw['IR-WSWFr'], errors='coerce')
dfw['IR-IrTot'] = pd.to_numeric(dfw['IR-IrTot'], errors='coerce')
dfw = dfw[dfw['IR-IrTot'] > 0]
dfw.loc[:,'wu_ratio'] = (dfw['IR-WGWFr'] > dfw['IR-WSWFr']).astype(int)
dfw.loc[:,'gwu_rate'] = dfw['IR-WGWFr'] / dfw['IR-IrTot']
dfw.loc[:,'swu_rate'] = dfw['IR-WSWFr'] / dfw['IR-IrTot']
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers
dfw = dfw[['FIPS','year','gwu_rate','swu_rate']]
dfw['FIPS'] = dfw['FIPS'].astype(int)
dfw

,FIPS,year,gwu_rate,swu_rate
0,1001,2009,1.610738,0.255034
1,1003,2009,2.236618,0.428832
2,1005,2009,0.166667,0.498148
3,1007,2009,0.214286,0.428571
4,1009,2009,0.508475,0.779661
...,...,...,...,...
46609,56037,2023,0.416110,3.546957
46610,56039,2023,0.000000,6.989101
46611,56041,2023,0.573282,4.887083
46612,56043,2023,0.207130,6.698673


In [ ]:
finsure = '<DATA_ROOT>/InsuranceData/InsuranceData.csv'
df_insure = pd.read_csv(finsure)
df_insure['FIPS'] = df_insure['State_Code'] * 1000 + df_insure['County_Code']
cols = ['FIPS', 'Commodity_Name', 'Year_Identifier','Cause_of_Loss_Code', 'Cause_of_Loss_Desc']
# cols = ['FIPS', 'Commodity_Name', 'Year_Identifier','Cause_of_Loss_Code', 'Cause_of_Loss_Desc','Subsidy','Liability']
df_insure = df_insure[cols]
df_insure = df_insure.rename(columns={'Commodity_Name':'Crop','Year_Identifier':'year'})
cond1 = df_insure['Cause_of_Loss_Code'] != 'XX'
df_insure = df_insure[cond1]
df_insure['Cause_of_Loss_Code'] = df_insure['Cause_of_Loss_Code'].astype(int)
df_insure = df_insure.groupby(['FIPS','Crop','year','Cause_of_Loss_Code']).first().reset_index()
df_insure

<ipython-input-45-c2fa01b3c2d3>:2: DtypeWarning: Columns (8,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_insure = pd.read_csv(finsure)


,FIPS,Crop,year,Cause_of_Loss_Code,Cause_of_Loss_Desc
0,1001,CORN,2008,11,Drought
1,1001,CORN,2009,1,Decline in Price
2,1001,CORN,2009,11,Drought
3,1001,CORN,2009,12,Heat
4,1001,CORN,2009,31,Excess Moisture/Precipitation/Rain
...,...,...,...,...,...
400252,56045,WHEAT,2016,1,Decline in Price
400253,56045,WHEAT,2016,11,Drought
400254,56045,WHEAT,2016,42,Freeze
400255,56045,WHEAT,2017,1,Decline in Price


In [ ]:
fcold = '<DATA_ROOT>/ClimateIndex/ColdSpell.csv'
df_cold = pd.read_csv(fcold)
cols = ['FIPS','Year','Thr-2', 'Thr0', 'Thr2', 'Thr4', 'Thr6',
        'Thr8', 'Thr10', 'Thr12', 'Thr14', 'Thr16', 'Thr18']
df_cold = df_cold[cols]
df_cold = df_cold.dropna(subset=['FIPS', 'Year'])
df_cold['FIPS'] = df_cold['FIPS'].astype(int)
df_cold['Year'] = df_cold['Year'].astype(int)
df_cold = df_cold.rename(columns={'Year':'year'})

In [ ]:
df_cold.columns

Index(['FIPS', 'year', 'Thr-2', 'Thr0', 'Thr2', 'Thr4', 'Thr6', 'Thr8',
       'Thr10', 'Thr12', 'Thr14', 'Thr16', 'Thr18'],
      dtype='object')

In [ ]:
fdrought = '<DATA_ROOT>/ClimateIndex/list_droughtFeatures.csv'
df_drought = pd.read_csv(fdrought)
df_drought = df_drought.dropna(subset=['Fips', 'Year'])
df_drought['Fips'] = df_drought['Fips'].astype(int)
df_drought['Year'] = df_drought['Year'].astype(int)
df_drought = df_drought.rename(columns={'Fips':'FIPS'})
df_drought = df_drought.rename(columns={'Year':'year'})

In [ ]:
df_drought.columns

Index(['FIPS', 'year', 'pdsi-1', 'pdsi-2', 'pdsi', 'spei180d-1', 'spei180d-2',
       'spei180d', 'spei1y-1', 'spei1y-2', 'spei1y', 'spei270d-1',
       'spei270d-2', 'spei270d', 'spei90d-1', 'spei90d-2', 'spei90d',
       'spi180d-1', 'spi180d-2', 'spi180d', 'spi1y-1', 'spi1y-2', 'spi1y',
       'spi270d-1', 'spi270d-2', 'spi270d', 'spi90d-1', 'spi90d-2', 'spi90d',
       'spei14d-1', 'spei14d-2', 'spei14d', 'spei30d-1', 'spei30d-2',
       'spei30d', 'spi14d-1', 'spi14d-2', 'spi14d', 'spi30d-1', 'spi30d-2',
       'spi30d'],
      dtype='object')

In [ ]:
fprcp = '<DATA_ROOT>/ClimateIndex/prcp.csv'
df_prcp = pd.read_csv(fprcp)
df_prcp = df_prcp.dropna(subset=['FIPS', 'Year'])
df_prcp['FIPS'] = df_prcp['FIPS'].astype(int)
df_prcp['Year'] = df_prcp['Year'].astype(int)
df_prcp = df_prcp.drop(columns=['State','County'])
df_prcp = df_prcp.rename(columns={'Year':'year'})

In [ ]:
df_prcp.columns

Index(['FIPS', 'year', 'dpi1', 'dpi5', 'dpi10', 'dpi20', 'dover1', 'dover5',
       'dover10', 'dover20'],
      dtype='object')

In [ ]:
fvpd = '<DATA_ROOT>/ClimateIndex/new_vpdfeatures.csv'
df_vpd = pd.read_csv(fvpd)
df_vpd = df_vpd.dropna(subset=['FIPS', 'Year'])
df_vpd['FIPS'] = df_vpd['FIPS'].astype(int)
df_vpd['Year'] = df_vpd['Year'].astype(int)
df_vpd = df_vpd.drop(columns=['State','County'])
df_vpd = df_vpd.rename(columns={'Year':'year'})

In [ ]:
df_vpd#.iloc[0:10000]

,FIPS,year,VPD-Mean,VPD-1,VPD-2
0,1001,2008,1.145214,125.0,1.0
1,1001,2009,0.947340,68.0,0.0
2,1001,2010,1.310861,147.0,7.0
3,1001,2011,1.690091,162.0,61.0
4,1001,2012,1.454901,157.0,19.0
...,...,...,...,...,...
46615,56045,2018,1.053937,93.0,14.0
46616,56045,2019,0.948979,81.0,8.0
46617,56045,2020,1.366875,117.0,44.0
46618,56045,2021,1.438504,123.0,51.0


In [ ]:
fwarm = '<DATA_ROOT>/ClimateIndex/WarmSpell.csv'
df_warm = pd.read_csv(fwarm)
df_warm = df_warm.dropna(subset=['FIPS', 'Year'])
df_warm['FIPS'] = df_warm['FIPS'].astype(int)
df_warm['Year'] = df_warm['Year'].astype(int)
df_warm = df_warm.drop(columns=['State','County'])
df_warm = df_warm.rename(columns={'Year':'year'})

In [ ]:
df_warm.columns

Index(['FIPS', 'year', 'Thr26', 'Thr27', 'Thr28', 'Thr29', 'Thr30', 'Thr31',
       'Thr32', 'Thr33', 'Thr34', 'Thr35', 'Thr36'],
      dtype='object')

In [ ]:
# df_fail , df_yield , df_svi , dfw , df_insure

In [ ]:
df_insure = df_insure.drop(columns=['Cause_of_Loss_Code'])
df_insure

,FIPS,Crop,year,Cause_of_Loss_Desc
0,1001,CORN,2008,Drought
1,1001,CORN,2009,Decline in Price
2,1001,CORN,2009,Drought
3,1001,CORN,2009,Heat
4,1001,CORN,2009,Excess Moisture/Precipitation/Rain
...,...,...,...,...
400252,56045,WHEAT,2016,Decline in Price
400253,56045,WHEAT,2016,Drought
400254,56045,WHEAT,2016,Freeze
400255,56045,WHEAT,2017,Decline in Price


In [ ]:
df_svi = df_svi[['FIPS','Crop','year','RPL_THEMES']]
df_cold = df_cold[['FIPS', 'year','Thr6', 'Thr10', 'Thr12', 'Thr16',]]
        # 'Thr0', 'Thr2', 'Thr4', 'Thr6', 'Thr8', 'Thr10', 'Thr12', 'Thr14', 'Thr16', 'Thr18']]
df_drought = df_drought[['FIPS', 'year','spei90d-1','spei30d-2']] # 'spi1y-2'
df_prcp = df_prcp[['FIPS', 'year', 'dover1', 'dover5']]
df_vpd = df_vpd[['FIPS', 'year','VPD-Mean','VPD-1','VPD-2']]
df_warm = df_warm[['FIPS', 'year','Thr28','Thr30','Thr31','Thr33','Thr35']]


In [ ]:
df_cold.columns

Index(['FIPS', 'year', 'Thr6', 'Thr10', 'Thr12', 'Thr16'], dtype='object')

# **Yield Analysis Regression All Categories**

In [ ]:
cond_irig = df_yield['prodn_practice_desc'] == 'ALL PRODUCTION PRACTICES'
df_yield = df_yield[cond_irig]
df_yield.loc[:,'prodn_practice_desc'] = 'ALL'
df_yield = df_yield.rename(columns={'commodity_desc':'Crop','prodn_practice_desc':'Irrigation Practice'})
df_yield['FIPS'] = df_yield['FIPS'].astype(int)

In [ ]:
# df_merge_yield_ins = pd.merge(df_yield,df_insure,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:m')
df_merge_yield_ins = df_yield
df_merge_yield_ins_svi = pd.merge(df_merge_yield_ins,df_svi,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:1')
df_merge_yield_ins_svi_gwu = pd.merge(df_merge_yield_ins_svi,dfw,on=['FIPS', 'year'],how='outer',validate='m:1')

df_merge_yield_ins_svi_gwu = df_merge_yield_ins_svi_gwu.dropna(subset='d_yield_standard')
condy = df_merge_yield_ins_svi_gwu.year < 2023
df_merge_yield_ins_svi_gwu = df_merge_yield_ins_svi_gwu[condy]

df_merge_clim = pd.merge(df_cold,df_drought,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_prcp,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_vpd,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_warm,on=['FIPS', 'year'])

df_merge = pd.merge(df_merge_yield_ins_svi_gwu,df_merge_clim,on=['FIPS', 'year'],validate='m:1')
df_merge = df_merge.dropna()

In [ ]:
Cause_of_Loss = [
    'Insects',	'Plant Disease',	'Frost',	'Hot Wind',
    'Wildlife',	'Freeze',	'Cold Winter',	'Wind/Excess Wind',
    'Flood',	'Hail',	'Heat',	'Cold Wet Weather',	'Drought',
    'Decline in Price',	'Excess Moisture/Precipitation/Rain'
]
# df_merge = df_merge[df_merge['Cause_of_Loss_Desc'].isin(Cause_of_Loss)]
# selected_cause = Cause_of_Loss[-3]
# cond_loss = df_merge['Cause_of_Loss_Desc'] == selected_cause
# df_merge_yield = df_merge[cond_loss]
df_merge_yield = df_merge
# print(len(df_merge_yield))
# df_merge_yield.groupby('Crop').count().FIPS

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

selected_crop = 'WHEAT' # 'WHEAT' , 'CORN' , 'SOYBEANS'
cond_crop = df_merge_yield.Crop == selected_crop
df = df_merge_yield[cond_crop].drop(columns=['FIPS','Crop','Irrigation Practice','year',
                                            #  'Cause_of_Loss_Desc',
                                              # 'spei30d-2','Thr28', 'Thr30', 'Thr31', 'Thr33','dover5',
                                              # 'VPD-Mean', 'VPD-1','Thr6', 'Thr10', 'Thr16',
                                              # 'VPD-Mean', 'VPD-1', 'Thr28', 'Thr30', 'Thr31', 'Thr33',
                                              # 'Thr6', 'Thr10', 'Thr12', 'spei30d-2','dover5',
                                              # 'VPD-Mean', 'VPD-1', 'Thr28', 'Thr30', 'Thr31', 'Thr33',
                                              # 'Thr6', 'Thr10', 'Thr12', 'spei90d-1','dover1',
                                              ])
# df = df_merge_yield.drop(columns=['FIPS','Crop','Irrigation Practice','year','Cause_of_Loss_Desc'])
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# df_scaled

In [ ]:
import numpy as np
import random
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df_scaled.drop(columns=['d_yield_standard'])
y = df_scaled['d_yield_standard']
r2_test = 0
r2_train = 0
rst = 0
for rs in range(1,2):
  rs = 108
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=rs) #

  params = {'max_depth': 20,
            'learning_rate': 0.04,
            'subsample': 0.8,
            'n_estimators': 120,
            'min_child_weight': 25,
              }
  xgb_model = xgb.XGBRegressor(**params)
  xgb_model.fit(X_train, y_train)
  y_pred = xgb_model.predict(X_test)
  y_pred2 = xgb_model.predict(X_train)
  r2_ts = r2_score(y_test, y_pred)
  r2_tr = r2_score(y_train, y_pred2)

  if r2_ts > r2_test:
    r2_test = r2_ts
    r2_train = r2_tr
    rst = rs
    print(rst)
    print("R-squared_test:", r2_ts)
    print("R-squared_train:", r2_tr)

108
R-squared_test: 0.36895817405081666
R-squared_train: 0.6727741369676288


In [ ]:
print(rst , r2_test , r2_train)

108 0.36895817405081666 0.6727741369676288


In [ ]:
# backward moving average
# df.loc[('WHEAT'),:] = {
#     'random_state':141, 'max_depth':20, 'learning_rate':0.05, 'subsample': 0.75, 'n_estimators':80, 'min_child_weight':25}
# R-squared_test: 0.37613271454368424   R-squared_train: 0.6353061117376648
# df.loc[('CORN'),:] = {
#     'random_state':438, 'max_depth':15, 'learning_rate':0.05,  'subsample': 0.8, 'n_estimators':75, 'min_child_weight':20}
# R-squared_test: 0.32116880359500355   R-squared_train: 0.5857733967412931
# df.loc[('SOYBEANS'),:] = {
#     'random_state':755, 'max_depth':15, 'learning_rate':0.03,  'subsample': 0.7, 'n_estimators':80, 'min_child_weight':20}
# R-squared_test: 0.4037936276443199    R-squared_train: 0.5749004535244393

# center moving average
# df.loc[('WHEAT'),:] = {
#     'random_state':108, 'max_depth':20, 'learning_rate':0.04, 'subsample': 0.8, 'n_estimators':120, 'min_child_weight':25}
# R-squared_test: 0.37613271454368424   R-squared_train: 0.6353061117376648
# df.loc[('CORN'),:] = {
#     'random_state':438, 'max_depth':15, 'learning_rate':0.05,  'subsample': 0.8, 'n_estimators':75, 'min_child_weight':20}
# R-squared_test: 0.32116880359500355   R-squared_train: 0.5857733967412931
# df.loc[('SOYBEANS'),:] = {
#     'random_state':755, 'max_depth':15, 'learning_rate':0.03,  'subsample': 0.7, 'n_estimators':80, 'min_child_weight':20}
# R-squared_test: 0.4037936276443199    R-squared_train: 0.5749004535244393




In [ ]:
!pip install shap scikit-learn pandas

In [ ]:
import shap
import matplotlib.pyplot as plt
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X, y)
shap.summary_plot(shap_values, X,show=False)
plt.title(selected_crop)
out_path = '<DATA_ROOT>/Final_Exports_2024_09/Shap_Summary_All_Causes_All_Features_V1/'
plt.gcf().savefig(out_path + selected_crop + "_centMV" + ".png")
# plt.close()

In [ ]:
'VPD-Mean', 'VPD-1', 'Thr28', 'Thr30', 'Thr31', 'Thr33',
'Thr6', 'Thr10', 'Thr12', 'spei90d-1','dover1',

('Thr6', 'Thr10', 'Thr12', 'spei90d-1', 'dover1')

In [ ]:
out_path = '<DATA_ROOT>/Final_Exports_2024_09/Shap_Scatter_centMV_All_Causes_All_Features_V1/'
shap_values_df = pd.DataFrame(shap_values.values, columns=X.columns)

for factor in list(X.columns):
  percentiles = np.percentile(df_merge_yield[cond_crop][factor], [10, 90])
  default_size = plt.rcParams['lines.markersize'] ** 2
  half_size = default_size / 2
  plt.scatter(df_merge_yield[cond_crop].RPL_THEMES, shap_values[:,factor].values,
                                                    #shap_values[:,factor].values
              c=df_merge_yield[cond_crop][factor], s=half_size, cmap='autumn_r',
              vmin=percentiles[0], vmax=percentiles[1])
  plt.colorbar(label=f'{factor} feature value')
  plt.xlabel('RPL_THEMES feature value')
  plt.ylabel(f'{factor} shap value')
  plt.title(f'Scatter Plot of {factor}')
  plt.gcf().savefig(out_path + selected_crop + '_' + factor + ".png") # out_path
  plt.close()
